In [1]:
!pip install --no-deps transformers==4.57.1
!pip install --no-deps "huggingface-hub>=0.34.0,<1.0"
!pip install -U bitsandbytes --no-deps
!pip install -U bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 91.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 49.3 MB/s eta 0:00:00


In [2]:
import transformers, torch
print("transformers:", transformers.__version__)
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
import bitsandbytes as bnb
print("bitsandbytes:", bnb.__version__)

transformers: 4.57.1
torch: 2.10.0+cu128 | CUDA available: True
bitsandbytes: 0.50.0


In [3]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import types, json, gc, re, string
import numpy as np
import torch
from PIL import Image
from transformers import AutoProcessor, LlavaOnevisionForConditionalGeneration, BitsAndBytesConfig
from transformers.cache_utils import DynamicCache
from transformers.masking_utils import create_causal_mask, create_sliding_window_causal_mask
from transformers.modeling_outputs import BaseModelOutputWithPast
from transformers.models.qwen2.modeling_qwen2 import apply_rotary_pos_emb
from kaggle_secrets import UserSecretsClient

MODEL_ID = "llava-hf/llava-onevision-qwen2-7b-ov-hf"
CHECKPOINT_LAYERS = [8, 11, 14]
EARLY_WINDOW = list(range(2, 9))
CHECKPOINT1_LAYER = 8
DROP_FRAC = 0.5

METADATA_PATH = "/kaggle/input/notebooks/mdnahianhasannokib/ecovlm-data-prep/vqav2_subset/metadata.json"
IMAGES_DIR = "/kaggle/input/notebooks/mdnahianhasannokib/ecovlm-data-prep/vqav2_subset/images"

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=hf_token)

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
model = LlavaOnevisionForConditionalGeneration.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, attn_implementation="eager", device_map="cuda:0", token=hf_token,
)
model.eval()
print("num_hidden_layers:", model.config.text_config.num_hidden_layers)

IMAGE_TOKEN_ID = model.config.image_token_index

with open(METADATA_PATH, "r") as f:
    vqav2_metadata = json.load(f)
print("num samples:", len(vqav2_metadata))

def build_inputs_for_sample(sample):
    image = Image.open(f"{IMAGES_DIR}/{sample['image_file']}").convert("RGB")
    question = sample["question"]
    conversation = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question}]}]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    sample_inputs = processor(images=image, text=prompt, return_tensors="pt").to(model.device)
    return sample_inputs, question

def apply_checkpoint_stub(hidden_states, position_embeddings, causal_mask_mapping, survivor_idx):
    hidden_states = hidden_states[:, survivor_idx, :]
    cos, sin = position_embeddings
    cos = cos[..., survivor_idx, :]
    sin = sin[..., survivor_idx, :]
    position_embeddings = (cos, sin)
    new_mask_mapping = {}
    for key, mask in causal_mask_mapping.items():
        new_mask_mapping[key] = mask[:, :, survivor_idx, :][:, :, :, survivor_idx] if mask is not None else None
    return hidden_states, position_embeddings, new_mask_mapping

print("SETUP COMPLETE")


processor_config.json:   0%|          | 0.00/178 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

preprocessor_config.json: 0.00B [00:00, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

... (11 outputs left out)

model-00003-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/126 [00:00<?, ?B/s]

num_hidden_layers: 28
num samples: 1000
SETUP COMPLETE


In [4]:
NUMBER_WORD_MAP = {
    "none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
    "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
}

def normalize_answer(ans):
    ans = ans.lower().strip()
    ans = re.sub(r"\b(a|an|the)\b", " ", ans)
    ans = ans.translate(str.maketrans("", "", string.punctuation))
    ans = " ".join(ans.split())
    words = [NUMBER_WORD_MAP.get(w, w) for w in ans.split()]
    return " ".join(words)

def vqa_accuracy(prediction, answers_list):
    pred = normalize_answer(prediction)
    norm_answers = [normalize_answer(a) for a in answers_list]
    accs = []
    for i in range(len(norm_answers)):
        others = norm_answers[:i] + norm_answers[i+1:]
        accs.append(min(sum(1 for a in others if a == pred) / 3.0, 1.0))
    return sum(accs) / len(accs)

CURRENT_PROMPT_INPUT_IDS = None
attn_capture = {}

def compute_survivor_idx_mean(input_ids_full, drop_frac):
    is_visual = (input_ids_full[0] == IMAGE_TOKEN_ID).cpu().numpy()
    visual_idx = np.nonzero(is_visual)[0]
    text_idx = np.nonzero(~is_visual)[0]
    if len(visual_idx) == 0:
        return torch.arange(input_ids_full.shape[1], device=input_ids_full.device)
    scores = np.zeros(len(visual_idx), dtype=np.float64)
    n_layers_used = 0
    for layer_idx in EARLY_WINDOW:
        if layer_idx not in attn_capture:
            continue
        scores += attn_capture[layer_idx][:, visual_idx].mean(axis=0)
        n_layers_used += 1
    if n_layers_used > 0:
        scores /= n_layers_used
    n_drop = int(len(visual_idx) * drop_frac)
    order = np.argsort(scores)
    keep_mask = np.ones(len(visual_idx), dtype=bool)
    keep_mask[order[:n_drop]] = False
    survivor = np.sort(np.concatenate([text_idx, visual_idx[keep_mask]]))
    return torch.tensor(survivor, device=input_ids_full.device, dtype=torch.long)

def dap_forward_checkpoint1_cached(
    self, input_ids=None, attention_mask=None, position_ids=None,
    past_key_values=None, inputs_embeds=None, use_cache=None, cache_position=None, **kwargs,
):
    seq_len_in = input_ids.shape[1] if input_ids is not None else inputs_embeds.shape[1]
    is_prefill = seq_len_in > 1
    if inputs_embeds is None:
        inputs_embeds = self.embed_tokens(input_ids)
    if use_cache and past_key_values is None:
        past_key_values = DynamicCache(config=self.config)
    if cache_position is None:
        past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
        cache_position = torch.arange(past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device)
    if position_ids is None:
        position_ids = cache_position.unsqueeze(0)

    hidden_states = inputs_embeds
    position_embeddings = self.rotary_emb(hidden_states, position_ids)

    causal_mask_mapping = None
    if is_prefill:
        if not isinstance(causal_mask_mapping := attention_mask, dict):
            mask_kwargs = {"config": self.config, "input_embeds": inputs_embeds, "attention_mask": attention_mask,
                           "cache_position": cache_position, "past_key_values": past_key_values, "position_ids": position_ids}
            causal_mask_mapping = {"full_attention": create_causal_mask(**mask_kwargs)}
            if self.has_sliding_layers:
                causal_mask_mapping["sliding_attention"] = create_sliding_window_causal_mask(**mask_kwargs)

    for i, decoder_layer in enumerate(self.layers[: self.config.num_hidden_layers]):
        layer_mask = causal_mask_mapping[decoder_layer.attention_type] if is_prefill else None
        hidden_states = decoder_layer(
            hidden_states, attention_mask=layer_mask, position_ids=position_ids,
            past_key_values=past_key_values, use_cache=use_cache, cache_position=cache_position,
            position_embeddings=position_embeddings, **kwargs,
        )
        if is_prefill and i == CHECKPOINT1_LAYER:
            survivor_idx = compute_survivor_idx_mean(CURRENT_PROMPT_INPUT_IDS, DROP_FRAC)
            hidden_states, position_embeddings, causal_mask_mapping = apply_checkpoint_stub(
                hidden_states, position_embeddings, causal_mask_mapping, survivor_idx
            )

    hidden_states = self.norm(hidden_states)
    return BaseModelOutputWithPast(last_hidden_state=hidden_states, past_key_values=past_key_values if use_cache else None)

model.set_attn_implementation("sdpa")
print("DAP FUNCTIONS READY, attn impl:", model.config._attn_implementation)
 
 

DAP FUNCTIONS READY, attn impl: sdpa


In [5]:
!pip install --no-deps zeus-ml --quiet

from zeus.monitor import ZeusMonitor
import time
import pandas as pd

zeus_monitor = ZeusMonitor(gpu_indices=[0])
zeus_monitor.begin_window("idle_measurement")
time.sleep(15)
idle_result = zeus_monitor.end_window("idle_measurement")
idle_power_w = idle_result.total_energy / idle_result.time
print(f"idle_power_w: {idle_power_w:.4f}")

def generate_with_telemetry(sample_inputs, use_dap_pruning, run_id, sample):
    global CURRENT_PROMPT_INPUT_IDS
    if use_dap_pruning:
        CURRENT_PROMPT_INPUT_IDS = sample_inputs["input_ids"]
        attn_capture.clear()
        original_forward = model.model.language_model.forward
        model.model.language_model.forward = types.MethodType(dap_forward_checkpoint1_cached, model.model.language_model)

    window_name = f"gen_{sample['sample_idx']}_{run_id}_{use_dap_pruning}"
    zeus_monitor.begin_window(window_name)
    with torch.no_grad():
        output_ids = model.generate(**sample_inputs, max_new_tokens=32, do_sample=False, use_cache=True)
    measurement = zeus_monitor.end_window(window_name)

    if use_dap_pruning:
        model.model.language_model.forward = original_forward

    generated_ids = output_ids[:, sample_inputs["input_ids"].shape[1]:]
    predicted_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]

    energy_gross_j = measurement.total_energy
    elapsed_s = measurement.time
    energy_net_j = energy_gross_j - (idle_power_w * elapsed_s)

    return {
        "run_id": run_id, "sample_idx": sample["sample_idx"], "question_id": sample["question_id"],
        "image_id": sample["image_id"], "question": sample["question"], "predicted": predicted_text,
        "vqa_accuracy": vqa_accuracy(predicted_text, sample["answers"]),
        "energy_gross_j": energy_gross_j, "energy_net_j": energy_net_j,
        "avg_power_w": energy_gross_j / elapsed_s if elapsed_s > 0 else 0.0,
        "idle_power_w": idle_power_w, "elapsed_s": elapsed_s,
        "tokens_generated": generated_ids.shape[1], "timestamp": pd.Timestamp.now().isoformat(),
    }

print("TELEMETRY READY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 5.8 MB/s eta 0:00:00
[2026-07-28 17:44:51,184] [zeus.device.gpu.nvidia](nvidia.py:47) pynvml is available and initialized.
[2026-07-28 17:44:51,185] [zeus.device.cpu.rapl](rapl.py:137) RAPL is not supported on this CPU.
[2026-07-28 17:44:51,186] [zeus.monitor.energy](energy.py:209) Monitoring GPU indices [0].
[2026-07-28 17:44:51,186] [zeus.monitor.energy](energy.py:210) Monitoring CPU indices []
[2026-07-28 17:44:51,187] [zeus.utils.framework](framework.py:25) PyTorch with CUDA support is available.
idle_power_w: 27.7892
TELEMETRY READY


In [6]:
import os

N_SAMPLES = 1000
CHECKPOINT_EVERY = 25
CHECKPOINT_PATH = "llavaonevision7b_dap_telemetry_n1000_checkpoint.csv"
FINAL_PATH = "llavaonevision7b_dap_telemetry_n1000.csv"

if os.path.exists(CHECKPOINT_PATH):
    existing_df = pd.read_csv(CHECKPOINT_PATH)
    telemetry_rows = existing_df.to_dict("records")
    done_sample_indices = set(existing_df["sample_idx"].unique())
    start_i = len([s for s in vqav2_metadata[:N_SAMPLES] if s["sample_idx"] in done_sample_indices])
    print(f"Resuming: {len(telemetry_rows)} rows done, starting at sample {start_i}")
else:
    telemetry_rows = []
    start_i = 0
    print("Starting fresh run")

for i in range(start_i, N_SAMPLES):
    sample = vqav2_metadata[i]
    try:
        sample_inputs, _ = build_inputs_for_sample(sample)
        telemetry_rows.append(generate_with_telemetry(sample_inputs, use_dap_pruning=False, run_id=1, sample=sample))
        sample_inputs2, _ = build_inputs_for_sample(sample)
        telemetry_rows.append(generate_with_telemetry(sample_inputs2, use_dap_pruning=True, run_id=1, sample=sample))
        del sample_inputs, sample_inputs2
    except Exception as e:
        print(f"ERROR on sample {i} (sample_idx={sample['sample_idx']}): {e}")

    gc.collect()
    torch.cuda.empty_cache()

    if (i + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(telemetry_rows).to_csv(CHECKPOINT_PATH, index=False)
        print(f"{i + 1}/{N_SAMPLES} done — checkpoint saved ({len(telemetry_rows)} rows)")

telemetry_df = pd.DataFrame(telemetry_rows)
telemetry_df.to_csv(FINAL_PATH, index=False)
print(f"\nFINAL: {len(telemetry_df)} rows saved to {FINAL_PATH}")

is_pruned = telemetry_df.index % 2 == 1
print("Baseline: acc=%.4f energy_net_j=%.1f" % (
    telemetry_df[~is_pruned]["vqa_accuracy"].mean(), telemetry_df[~is_pruned]["energy_net_j"].mean()))
print("Pruned:   acc=%.4f energy_net_j=%.1f" % (
    telemetry_df[is_pruned]["vqa_accuracy"].mean(), telemetry_df[is_pruned]["energy_net_j"].mean()))
print("Energy reduction: %.1f%%" % (
    100 * (1 - telemetry_df[is_pruned]["energy_net_j"].mean() / telemetry_df[~is_pruned]["energy_net_j"].mean())))

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


Starting fresh run


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

25/1000 done — checkpoint saved (50 rows)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

50/1000 done — checkpoint saved (100 rows)


... (72 outputs left out)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

975/1000 done — checkpoint saved (1950 rows)


Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151645 for

1000/1000 done — checkpoint saved (2000 rows)

FINAL: 2000 rows saved to llavaonevision7b_dap_telemetry_n1000.csv
Baseline: acc=0.7979 energy_net_j=854.9
Pruned:   acc=0.7523 energy_net_j=576.3
Energy reduction: 32.6%
